# Raw Ingestion

## Import Helper Functions

In [1]:
import boto3
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime

from src import *

# while it is allowed to use the import above, it is advised to list out what we have imported
# from src.config import *
# from src.spark_sql_magic import sql
# from src.raw_utils import safe_file_type, checksum, get_target_path, get_spark_path

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Import Libraries and Start Session

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.types import *
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["raw_ingestion"])
        .getOrCreate()
)

## DDL: raw_ingestion_metadata

In [3]:
table_name = f"{CATALOG}.{RAW_FOLDER}.{RAW_METADATA_TABLE}"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        file_name STRING,
        file_type STRING,
        path STRING,
        size_bytes BIGINT,
        upload_ts TIMESTAMP,
        checksum STRING
    )
    USING ICEBERG
    PARTITIONED BY (days(upload_ts))
""")

print(f"{table_name} created.")

polaris.raw.raw_ingestion_metadata created.


## Create s3 Client (Minio)

In [4]:
s3 = boto3.client(
    "s3",
    endpoint_url="http://minio:9000",
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin",
)

## Function: Upload Raw File

In [5]:
def upload_raw_file(file_path:str) -> str:
    """
    Upload file to MinIO and log metadata to Iceberg.
    Preserves original format.
    """
    
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(file_path)

    # dedup check
    file_checksum = checksum(file_path)

    exists = (
        spark.table(table_name)
            .filter(F.col("checksum") == file_checksum)
            .limit(1)
            .count()
    )

    if exists:
        print(f"{file_path.name} already ingested.")
        return get_spark_path(file_path)

    # Upload to MinIO (raw format preserved)
    object_key = f"{RAW_FOLDER}/{file_path.name}"

    s3.upload_file(
        str(file_path),
        BUCKET,
        object_key
    )
    spark_path = f"s3a://{BUCKET}/{object_key}"

    log_ingestion_metadata(
        spark,
        table_name,
        file_path,
        spark_path,
        file_checksum,
    )

    return spark_path

In [6]:
# ----------------------------
# Path inside the container
# ----------------------------
RAW_DATA_DIR = Path("/opt/spark/datasets")

files = list(RAW_DATA_DIR.glob("*"))

if not files:
    print("No files found.")
else:
    uploaded_paths = []

    for file in files:
        spark_path = upload_raw_file(file)
        uploaded_paths.append(spark_path)

    print("\nUploaded paths:")
    for p in uploaded_paths:
        print(p)


Uploaded paths:
s3a://olist-ecommerce/raw/olist_customers_dataset.csv
s3a://olist-ecommerce/raw/olist_geolocation_dataset.csv
s3a://olist-ecommerce/raw/olist_orders_dataset.csv
s3a://olist-ecommerce/raw/olist_order_items_dataset.csv
s3a://olist-ecommerce/raw/olist_order_payments_dataset.csv
s3a://olist-ecommerce/raw/olist_order_reviews_dataset.csv
s3a://olist-ecommerce/raw/olist_products_dataset.csv
s3a://olist-ecommerce/raw/olist_sellers_dataset.csv
s3a://olist-ecommerce/raw/product_category_name_translation.csv


In [7]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop()  